In [ ]:
%matplotlib inline
from pandarallel import pandarallel
import torch
pandarallel.initialize(progress_bar=True)  
%matplotlib inline
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde


expr = pd.read_csv('.\input\expr.csv', index_col=0).values
coords = pd.read_csv('.\input\coordinates.csv', index_col=0).values
full_edges = torch.load('.\\input\\full_edges.pt')

In [ ]:
from model import aggregator as ag

gene_mins = np.min(expr, axis=0, keepdims=True)
gene_maxs = np.max(expr, axis=0, keepdims=True)

range_vals = gene_maxs - gene_mins
range_vals[range_vals == 0] = 1
expr_normalized = (expr - gene_mins) / range_vals
expr_tensor = torch.tensor(expr_normalized, dtype=torch.float32)
trained_model = ag.train_model(expr_tensor, full_edges, num_epochs=1000)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
z_local = torch.load(".\output\z_local.pt")
import numpy as np
import pandas as pd
import os
import torch
import matplotlib.pyplot as plt

device = torch.device('cpu')

print(f"读取到 {len(coords)} 个细胞的坐标数据")
x_coords = coords['normalized_x'].values
y_coords = coords['normalized_y'].values
plt.figure(figsize=(10, 8))
plt.scatter(x_coords, y_coords, s=5, alpha=0.5)
plt.title('细胞空间分布')
plt.xlabel('Normalized X')
plt.ylabel('Normalized Y')
plt.grid(True, linestyle='--', alpha=0.3)

xy = np.vstack([x_coords, y_coords])
density = gaussian_kde(xy)(xy)

density_quantiles = np.quantile(density, [0.25, 0.5, 0.75])
print(f"密度分位数: Q1={density_quantiles[0]:.4f}, Q2={density_quantiles[1]:.4f}, Q3={density_quantiles[2]:.4f}")

num_blocks_x = 10
num_blocks_y = 10
total_blocks = num_blocks_x * num_blocks_y

x_bins = np.quantile(x_coords, np.linspace(0, 1, num_blocks_x + 1))
y_bins = np.quantile(y_coords, np.linspace(0, 1, num_blocks_y + 1))

x_bins[0] = np.min(x_coords)
x_bins[-1] = np.max(x_coords)
y_bins[0] = np.min(y_coords)
y_bins[-1] = np.max(y_coords)

print(f"X划分点: {np.round(x_bins, 4)}")
print(f"Y划分点: {np.round(y_bins, 4)}")

region_ids = np.zeros(len(coords), dtype=int)

for i in range(num_blocks_x):
    for j in range(num_blocks_y):
        
        x_mask = (x_coords >= x_bins[i]) & (x_coords < x_bins[i + 1])
        y_mask = (y_coords >= y_bins[j]) & (y_coords < y_bins[j + 1])

        block_mask = x_mask & y_mask

        region_ids[block_mask] = i * num_blocks_y + j

region_ids[x_coords == x_bins[-1]] = (num_blocks_x - 1) * num_blocks_y + (num_blocks_y - 1)
region_ids[y_coords == y_bins[-1]] = (num_blocks_x - 1) * num_blocks_y + (num_blocks_y - 1)

region_cell_counts = np.bincount(region_ids, minlength=total_blocks)
empty_regions = np.sum(region_cell_counts == 0)
print(f"\n区域统计: 总区域数={total_blocks}, 有细胞区域数={total_blocks - empty_regions}, 空区域数={empty_regions}")

region_ids_tensor = torch.tensor(region_ids, device=device)

z_fused = torch.zeros((total_blocks, z_local.shape[1]),
                      device=device)

for region_id in range(total_blocks):
    mask = (region_ids_tensor == region_id)
    if torch.any(mask):
        z_fused[region_id] = torch.mean(z_local[mask], dim=0)
    else:
        
        region_x = region_id // num_blocks_y
        region_y = region_id % num_blocks_y

        for dist in range(1, max(num_blocks_x, num_blocks_y)):
            neighbors = []
            for dx in range(-dist, dist + 1):
                for dy in range(-dist, dist + 1):
                    nx, ny = region_x + dx, region_y + dy
                    if 0 <= nx < num_blocks_x and 0 <= ny < num_blocks_y:
                        neighbor_id = nx * num_blocks_y + ny
                        if region_cell_counts[neighbor_id] > 0:
                            neighbors.append(neighbor_id)
            if neighbors:
                
                neighbor_ids = torch.tensor(neighbors, device=device)
                z_fused[region_id] = torch.mean(z_fused[neighbor_ids], dim=0)
                print(f"区域 {region_id} 使用 {len(neighbors)} 个邻居填充")
                break
        else:
            
            z_fused[region_id] = torch.mean(z_local, dim=0)
            print(f"区域 {region_id} 使用全局平均值填充")

z_fused_np = z_fused.cpu().numpy()
np.save('output/z_fused.npy', z_fused_np)
print(f"保存融合特征到 output/z_fused.npy, 形状: {z_fused_np.shape}")

region_df = pd.DataFrame({
    'cell_id': coords.iloc[:, 0],
    'region_id': region_ids
})
region_df.to_csv('output/region_ids.csv', index=False)
print(f"保存区域ID到 output/region_ids.csv")

region_info = []
for region_id in range(total_blocks):
    i = region_id // num_blocks_y
    j = region_id % num_blocks_y

    region_mask = (region_ids == region_id)
    cell_count = region_cell_counts[region_id]

    region_info.append({
        'region_id': region_id,
        'grid_x': i,
        'grid_y': j,
        'x_min': x_bins[i],
        'x_max': x_bins[i + 1],
        'y_min': y_bins[j],
        'y_max': y_bins[j + 1],
        'cell_count': cell_count,
        'mean_x': np.mean(x_coords[region_mask]) if cell_count > 0 else (x_bins[i] + x_bins[i + 1]) / 2,
        'mean_y': np.mean(y_coords[region_mask]) if cell_count > 0 else (y_bins[j] + y_bins[j + 1]) / 2
    })

region_df = pd.DataFrame(region_info)
region_df.to_csv('output/region_info.csv', index=False)
print("保存区域信息到 output/region_info.csv")